# Анализ кросс-продаж онлайн-курсов

Учебный проект. Есть история покупок онлайн-курсов. Хочу найти курсы, которые часто покупают одни и те же люди, и на основе этого предлагать пользователю ещё пару курсов к уже выбранному.

**Стек:** Python, pandas.

## Данные

- Источник: синтетический датасет, получен как учебное задание на курсе.
- Файлы:
  - `all_courses.csv` — список курсов (127 штук).
  - `purchased_courses.csv` — покупки пользователей (34 172 записи).
- В файле покупок есть только `user_id` и `resource_id`. Нет данных о том, что покупалось в одном заказе, и нет дат покупок.

*Примечание:* Также в исходных материалах присутствовал файл `popularity_rank.csv` (рейтинг популярности). Он был исключен из обработки, так как методология его расчета неизвестна, а для решения задачи прямого подсчета частоты пар он не требуется.

## Ограничения

- Данные синтетические, поэтому выводы условные. Это учебный проект, 
  а не реальное исследование бизнеса.
- В данных нет информации о заказах: неизвестно, какие курсы покупались 
  вместе в одной покупке. Поэтому смотрю не на состав заказов, а на то, 
  какие курсы встречаются у одних и тех же пользователей.
- Рекомендации строятся только по частоте пар. Популярный курс попадает 
  в пары часто просто потому, что его многие покупают, а не потому, 
  что он связан с другими. В этом проекте проблему не решаю — для этого 
  нужны дополнительные метрики, которые здесь не используются.
- В данных у каждого пользователя минимум две покупки. В реальной 
  онлайн-школе обычно большинство покупает один курс и уходит, так что 
  структура данных здесь нетипичная.
- Пары курсов рассматриваю как неупорядоченные: (A, B) — то же самое, 
  что (B, A).

## Структура ноутбука

1. Загрузка и первичный осмотр данных
2. Очистка данных
3. Подготовка данных
4. Частота пар курсов
5. Формирование рекомендаций
6. Выводы
7. Что бы я улучшил

## Импорты и скачивание файлов

In [1]:
# Стандартная библиотека
from collections import Counter
from itertools import combinations
import os
from pathlib import Path
import urllib.request

# Сторонние библиотеки
import pandas as pd

# Автоматическое скачивание исходных данных с GitHub в сессию Google Colab
os.makedirs('data', exist_ok=True)

urls = {
    'data/all_courses.csv': 'https://githubusercontent.com',
    'data/purchased_courses.csv': 'https://githubusercontent.com'
}

for path, url in urls.items():
    if not os.path.exists(path):
        print(f"Скачивание {path}...")
        urllib.request.urlretrieve(url, path)
        print(f"Файл {path} успешно загружен.")
    else:
        print(f"Файл {path} уже существует локально.")

Файл data/all_courses.csv уже существует локально.
Файл data/purchased_courses.csv уже существует локально.


## 1. Загрузка и первичный осмотр данных

In [2]:
data_dir = Path("data")

for f in sorted(data_dir.glob("*.csv")):
    print(f"{f.name:40s} {f.stat().st_size / 1024:8.1f} KB")

all_courses.csv                               0.7 KB
purchased_courses.csv                       421.0 KB


In [3]:
for f in sorted(data_dir.glob("*.csv")):
    print(f"===== {f.name} =====")
    with open(f, encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i >= 5:
                break
            print(line.rstrip())

===== all_courses.csv =====
"resource_id"
809
1145
908
814
===== purchased_courses.csv =====
"user_id","resource_id"
1010882,490
906674,357
160494,507
749529,489


In [4]:
courses = pd.read_csv("data/all_courses.csv")
purchases = pd.read_csv("data/purchased_courses.csv")

# Быстрый осмотр
for name, df in [("courses", courses), ("purchases", purchases)]:
    print(f"===== {name} =====")
    print(f"shape: {df.shape}")
    print(df.dtypes)
    print(df.head(3))
    print()

===== courses =====
shape: (127, 1)
resource_id    int64
dtype: object
   resource_id
0          809
1         1145
2          908

===== purchases =====
shape: (34172, 2)
user_id        int64
resource_id    int64
dtype: object
   user_id  resource_id
0  1010882          490
1   906674          357
2   160494          507



### Оптимизация типов
Pandas по умолчанию загружает целые числа как `int64`, даже если значения 
помещаются в меньший тип. Для учебного проекта это неважно, но в реальной 
работе с большими файлами лишняя память может стать проблемой. Понижу 
разрядность целочисленных колонок и проверю, насколько уменьшился расход.

Все три таблицы маленькие, экономия в килобайтах. Но приём рабочий: 
на реальных данных он даёт ощутимый выигрыш.

In [5]:
def optimize_ints(df, name):
    """Понижает разрядность int64-колонок. Меняет df на месте."""
    mem_before = df.memory_usage(deep=True).sum() / 1024
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    mem_after = df.memory_usage(deep=True).sum() / 1024
    print(f"{name}: {mem_before:.1f} KB → {mem_after:.1f} KB "
          f"(экономия {(1 - mem_after / mem_before) * 100:.0f}%)")

optimize_ints(courses, "courses")
optimize_ints(purchases, "purchases")

print("\nТипы после оптимизации:")
for name, df in [("courses", courses), ("purchases", purchases)]:
    print(f"\n{name}:")
    for col, dtype in df.dtypes.items():
        print(f"  {col}: {dtype}")

courses: 1.1 KB → 0.4 KB (экономия 66%)
purchases: 534.1 KB → 200.4 KB (экономия 62%)

Типы после оптимизации:

courses:
  resource_id: int16

purchases:
  user_id: int32
  resource_id: int16


### Осмотр данных

In [6]:
# Уникальные пользователи и курсы
print("Уникальных пользователей:", purchases["user_id"].nunique())
print("Уникальных курсов в purchases:", purchases["resource_id"].nunique())
print("Курсов в каталоге:", courses["resource_id"].nunique())

# Пересечения
courses_set = set(courses["resource_id"])
purchases_set = set(purchases["resource_id"])

print("\nКурсов в purchases, которых нет в каталоге:", len(purchases_set - courses_set))
print("Курсов в каталоге, которых нет в purchases:", len(courses_set - purchases_set))

# Покупки на пользователя
purchases_per_user = purchases.groupby("user_id")["resource_id"].nunique()
print("\nПокупок на пользователя (описательные статистики):")
print(purchases_per_user.describe())

# Пользователи с одной покупкой — пары не дадут
print("\nПользователей с 1 покупкой:", (purchases_per_user == 1).sum())

# Дубли (user_id, resource_id)
print("\nДубли (user_id, resource_id):", purchases.duplicated().sum())

Уникальных пользователей: 12656
Уникальных курсов в purchases: 126
Курсов в каталоге: 127

Курсов в purchases, которых нет в каталоге: 0
Курсов в каталоге, которых нет в purchases: 1

Покупок на пользователя (описательные статистики):
count    12656.000000
mean         2.692320
std          1.329538
min          2.000000
25%          2.000000
50%          2.000000
75%          3.000000
max         19.000000
Name: resource_id, dtype: float64

Пользователей с 1 покупкой: 0

Дубли (user_id, resource_id): 98


### Что видно в данных

- Пользователей: 12 656. У всех минимум 2 покупки.
  В реальной онлайн-школе обычно большинство покупает 
  один курс и больше не возвращается. Похоже, при генерации данных 
  заложили условие «минимум 2 покупки». Это удобно для анализа 
  (у каждого пользователя есть хотя бы одна пара курсов), но на реальных 
  данных картина была бы другой.
- Курсов в покупках: 126. Один курс из каталога никто не купил.
- Дублей (user_id, resource_id): 98 из 34 172. Это мало, но уберу — они 
  не должны считаться дважды.

## 2. Очистка данных

In [7]:
purchases_clean = purchases.drop_duplicates().reset_index(drop=True)

print("Строк до очистки:", len(purchases))
print("Строк после очистки:", len(purchases_clean))
print("Удалено дублей:", len(purchases) - len(purchases_clean))

Строк до очистки: 34172
Строк после очистки: 34074
Удалено дублей: 98


In [8]:
# Сколько строк (покупок, включая повторные)
purchases_per_user_rows = purchases_clean.groupby("user_id").size()

# Сколько разных курсов у пользователя
purchases_per_user_unique = purchases_clean.groupby("user_id")["resource_id"].nunique()

print("Пользователей с 1 строкой покупки:", (purchases_per_user_rows == 1).sum())
print("Пользователей с 1 уникальным курсом:", (purchases_per_user_unique == 1).sum())

Пользователей с 1 строкой покупки: 0
Пользователей с 1 уникальным курсом: 0


### Итог

Убрал 98 дублей. После очистки у всех пользователей по-прежнему 
минимум 2 разных курса — значит, пары для анализа есть у каждого, 
никого отбрасывать не нужно.

## 3. Подготовка данных

Теперь из таблицы покупок соберу для каждого пользователя список курсов, которые он купил. Это нужно, чтобы на следующем шаге составить пары курсов для каждого пользователя.

In [9]:
# Для каждого пользователя собираю список его курсов
user_courses = (
    purchases_clean
    .groupby("user_id")["resource_id"]
    .apply(list)
    .apply(sorted)
)

print("Пользователей:", len(user_courses))
print("\nПервые 5:")
print(user_courses.head())

Пользователей: 12656

Первые 5:
user_id
51            [516, 1099]
6117     [356, 357, 1125]
10275         [553, 1147]
10457         [361, 1138]
17166          [356, 357]
Name: resource_id, dtype: object


### Итог

Для каждого из 12 656 пользователей есть отсортированный список 
курсов, которые он купил.

## 4. Частота пар курсов

Теперь для каждого пользователя составлю все возможные пары его курсов. 
Например, из [10, 20, 30] получатся пары (10, 20), (10, 30), (20, 30). 
Затем посчитаю, у скольких пользователей встречается каждая пара.

In [10]:
# Собираю все пары курсов по всем пользователям
all_pairs = []
for courses_list in user_courses:
    all_pairs.extend(combinations(courses_list, 2))

print("Всего пар (с повторами):", len(all_pairs))

# Считаю, сколько раз встречается каждая пара
pair_counts = Counter(all_pairs)

print("Уникальных пар:", len(pair_counts))
print("\nТоп-5 самых частых пар:")
for pair, count in pair_counts.most_common(5):
    print(pair, "—", count, "раз")

Всего пар (с повторами): 40017
Уникальных пар: 3989

Топ-5 самых частых пар:
(551, 566) — 797 раз
(515, 551) — 417 раз
(489, 551) — 311 раз
(523, 551) — 304 раз
(566, 794) — 290 раз


In [11]:
pair_freq = pd.Series(pair_counts).sort_values(ascending=False)

print("Описательные статистики частоты пар:")
print(pair_freq.describe())

print("\nСколько пар встречается 1 раз:", (pair_freq == 1).sum())
print("Сколько пар встречается 2 раза:", (pair_freq == 2).sum())
print("Сколько пар встречается 3 раза:", (pair_freq == 3).sum())
print("Сколько пар встречается 5+ раз:", (pair_freq >= 5).sum())
print("Сколько пар встречается 10+ раз:", (pair_freq >= 10).sum())

Описательные статистики частоты пар:
count    3989.000000
mean       10.031838
std        26.355998
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       797.000000
dtype: float64

Сколько пар встречается 1 раз: 1185
Сколько пар встречается 2 раза: 578
Сколько пар встречается 3 раза: 366
Сколько пар встречается 5+ раз: 1601
Сколько пар встречается 10+ раз: 928


### Что видно

- Всего пар (с повторами): 40 017. Уникальных пар: 3 989.
- Самая частая пара — (551, 566), 797 раз.
- Распределение частот сильно скошено: 30% пар встречаются всего 1 раз, 
  ещё 14% — 2 раза. Такие пары ненадёжны для рекомендаций.
- Важное ограничение метода: частота пары не означает, что курсы 
  связаны. Популярный курс попадает в пары часто просто потому, 
  что его многие покупают. В этом учебном проекте я не решаю эту 
  проблему — для этого нужны дополнительные метрики (например, lift), 
  которые здесь не используются. Поэтому рекомендации стоит 
  воспринимать как «что часто покупают вместе», а не «что связано».

### Решение по порогу

Беру пары с частотой ≥ 10. Это отсекает пары, которые встретились 
у 1–2 пользователей и ничего не говорят о связи. После фильтра 
останется 928 пар — достаточно для анализа, но без очевидного мусора.

## 5. Формирование рекомендаций

Теперь для каждого курса найду две рекомендации — курсы, которые 
чаще всего покупают вместе с ним. Работаю с парами, прошедшими 
фильтр из раздела 4 (частота ≥ 10).

Для каждого курса смотрю все пары, где он участвует, сортирую их 
по частоте и беру два первых. Если пар меньше двух — рекомендаций 
будет меньше.

In [12]:
# Таблица пар с частотами (из pair_counts раздела 4)
pairs_df = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in pair_counts.items()],
    columns=["course_a", "course_b", "freq_ab"]
)

# Фильтр из раздела 4: частота ≥ 10
pairs_filtered = pairs_df[pairs_df["freq_ab"] >= 10].copy()

print("Всего пар:", len(pairs_df))
print("После фильтра freq ≥ 10:", len(pairs_filtered))

Всего пар: 3989
После фильтра freq ≥ 10: 928


In [13]:
all_courses = sorted(purchases_clean["resource_id"].unique())

recommendations = []

for course in all_courses:
    # Собираем всех «соседей» этого курса из пар
    # Использую iterrows для читаемости. На наших объёмах (928 пар) сработает быстро. На больших данных заменил бы на векторизованный вариант — см. «Что бы я улучшил».
    candidates = []
    for _, row in pairs_filtered.iterrows():
        if row["course_a"] == course:
            candidates.append((row["course_b"], row["freq_ab"]))
        elif row["course_b"] == course:
            candidates.append((row["course_a"], row["freq_ab"]))
    
    # Сортируем по частоте, от большей к меньшей
    candidates.sort(key=lambda x: x[1], reverse=True)
    
    # Берём до двух
    rec_1 = candidates[0][0] if len(candidates) > 0 else None
    rec_2 = candidates[1][0] if len(candidates) > 1 else None
    
    recommendations.append({
        "course": course,
        "rec_1": rec_1,
        "rec_2": rec_2,
    })

recommendations_df = pd.DataFrame(recommendations, dtype='Int16')

In [14]:
print("Курсов всего:", len(recommendations_df))
print("Курсов с двумя рекомендациями:", recommendations_df["rec_2"].notna().sum())
print("Курсов с одной рекомендацией:", (recommendations_df["rec_1"].notna() & recommendations_df["rec_2"].isna()).sum())
print("Курсов без рекомендаций:", recommendations_df["rec_1"].isna().sum())

print("\nПервые 10 строк:")
recommendations_df.head(10)

Курсов всего: 126
Курсов с двумя рекомендациями: 97
Курсов с одной рекомендацией: 7
Курсов без рекомендаций: 22

Первые 10 строк:


,course,rec_1,rec_2
0,356,571,357
1,357,571,356
2,358,570,752
3,359,570,358
4,360,745,516
5,361,551,1138
6,362,363,<NA>
7,363,511,562
8,364,<NA>,<NA>
9,365,<NA>,<NA>


## 6. Выводы

**Что нашли:**
- Самая частая пара — (551, 566): её купили 797 пользователей.
- Всего 3 989 уникальных пар. Но 44% из них встречаются у 1–2 
  пользователей — это скорее случайные совпадения, чем связи.
- После фильтра (частота ≥ 10) осталось 928 пар.

**Что построили:**
- Таблицу рекомендаций: для каждого из 126 курсов — до двух курсов, 
  которые часто покупают вместе с ним.
- 97 курсов (77%) получили две рекомендации, 7 курсов (6%) — одну, 
  22 курса (17%) остались без рекомендаций.

**Ограничения:**
- Данные синтетические. Реальные выводы для бизнеса делать нельзя.
- Метод — частота пар. Популярный курс часто попадает в пары просто 
  потому, что его многие покупают. В этом проекте проблему не решаю — 
  для этого нужны дополнительные метрики (confidence, lift), которые 
  здесь не используются.
- У 22 курсов нет рекомендаций: для них нет достаточно частых пар. 
  В реальной системе для них нужен отдельный подход.

## 7. Что бы я улучшил

**1. Заменить `iterrows` на векторизованные операции.**
В коде формирования рекомендаций я использую `iterrows` — это 
медленно на больших данных. На наших 928 парах работает быстро, 
но в реальной системе с миллионами пар это стало бы узким местом.

**2. Добавить метрики силы связи.**
Сейчас рекомендации строятся только по частоте пар. Это значит, 
что в топ попадают пары с популярными курсами — просто потому, 
что их многие покупают. Чтобы отделить «настоящую связь» 
от «популярности», нужны дополнительные метрики: confidence 
(доля покупателей A, которые берут ещё и B) и lift (во сколько 
раз эта доля больше случайной). Я их не использую в этом 
проекте, но знаю, что они решают эту проблему.

**3. Fallback для курсов без пар.**
Сейчас 77 курсов из 126 не получают рекомендаций — для них нет 
пар с частотой ≥ 10. В реальной системе можно было бы 
рекомендовать популярные курсы как запасной вариант, но 
это требует отдельного решения: показывать «топ продаж» 
или оставлять блок пустым.

**4. Цены и маржа.**
У нас нет цен и маржи по курсам. Если бы были — можно было бы 
оптимизировать рекомендации не по вероятности покупки, 
а по ожидаемой выгоде (вероятность × маржа). Сейчас это 
невозможно.

**5. Контентные признаки.**
У нас нет названий, тем, уровней курсов. Если бы были — 
можно было бы строить рекомендации не только по со-покупкам, 
но и по схожести содержания. Это помогло бы для курсов, 
у которых мало пар.

**6. A/B-тест на реальных данных.**
Наши данные синтетические, проверить эффект от рекомендаций 
нельзя. В реальной системе нужно было бы провести A/B-тест: 
показать рекомендации одной группе пользователей и не показать 
другой — и сравнить, выросла ли конверсия.